In [1]:
!pip install sentencepiece tiktoken evaluate
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 75.2 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1


In [2]:
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, BartForConditionalGeneration, BartTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset
import torch
import evaluate

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
splits = ['Train', 'Val', 'Test']
dfs = {}

for split in splits:
  path = f'/content/drive/MyDrive/266_Final/data/{split}/data.parquet'
  dfs[split] = pd.read_parquet(path)
  print(f'{split} shape: {dfs[split].shape}')

train_df, val_df, test_df = dfs['Train'], dfs['Val'], dfs['Test']

Train shape: (53492, 6)
Val shape: (6686, 6)
Test shape: (6687, 6)


In [5]:
train_df.head()

,question,answer,context,pairs,positive_sentences,image
0,geno hayes and nick roach both played which po...,no answer,Position Player $ Free agency tag Date signed ...,"[{'label': 0, 'query': 'geno hayes and nick ro...",0,204_783_page.png
1,What is the number of bronze for Canada and si...,no answer,"Pos Team Time Gold Switzerland (Reto Götschi, ...","[{'label': 0, 'query': 'What is the number of ...",0,11338408_page.png
2,"What is Date, when Opponent is ""Philadelphia 7...",february 16,# Date H/A/N Opponent W/L Score Record 40 Janu...,"[{'label': 1, 'query': 'What is Date, when Opp...",1,16946097_page.png
3,Which Class has a Weight of 203?,sr,Position Number Name Height Weight Class Homet...,"[{'label': 1, 'query': 'Which Class has a Weig...",29,14624447_page.png
4,Who had 0 total votes in the purple team?,eliminated week 12 - the walk,Members Teammate Couples Team Blue Red 1 Blue ...,"[{'label': 1, 'query': 'Who had 0 total votes ...",1,18426579_page.png


In [6]:
train_df['context'][45000]

'Seat State Majority Member Party HINKLER Qld 00.0 Paul Neville NPA SOLOMON NT 00.1 Dave Tollner Lib ADELAIDE SA 00.2 Hon Trish Worth Lib CANNING WA 00.4 Don Randall Lib DOBELL NSW 00.4 Ken Ticehurst Lib PARRAMATTA NSW 01.1 Ross Cameron Lib McEWEN Vic 01.2 Fran Bailey Lib PATERSON NSW 01.4 Bob Baldwin Lib HERBERT Qld 01.6 Peter Lindsay Lib RICHMOND NSW 01.6 Hon Larry Anthony NPA DEAKIN Vic 01.7 Philip Barresi Lib EDEN-MONARO NSW 01.7 Gary Nairn Lib HINDMARSH SA 01.9 Hon Christine Gallus Lib White (Democrat) 6,000 06.0% Smith (Labor) 45,000 45.0% Jones (Liberal) 35,000 35.0% Johnson (Green) 10,000 10.0% Davies (Ind) 4,000 04.0%'

In [7]:
model = "microsoft/tapex-base"

In [8]:
tokenizer = BartTokenizer.from_pretrained(model, use_fast=False)
model = BartForConditionalGeneration.from_pretrained(model)

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/988 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
def preprocess_function(examples):
  inputs = [
        f"{q.strip()} {c.strip()}"
        for q, c in zip(examples["question"], examples["context"])
  ]

  model_inputs = tokenizer(
        inputs,
        max_length=1024,
        padding=False,
        truncation=True,
    )

  labels = tokenizer(
        text_target=examples["answer"],
        max_length=1024,
        padding=False,
        truncation=True,
    )

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs




In [10]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [11]:
tokenized_train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

tokenized_test_dataset = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/53492 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Map:   0%|          | 0/6687 [00:00<?, ? examples/s]

In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [13]:
tokenized_train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 53492
})

In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./tapex-base-paper",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=32,
    eval_strategy="epoch",
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
)

In [17]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.733920
2,No log,0.619866
3,31.236240,0.566449
4,31.236240,0.542229
5,14.899389,0.538325
6,14.899389,0.521204
7,14.899389,0.524748
8,10.700418,0.528277
9,10.700418,0.529115
10,8.648055,0.530032


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2090, training_loss=16.005410401558763, metrics={'train_runtime': 2687.494, 'train_samples_per_second': 199.04, 'train_steps_per_second': 0.778, 'total_flos': 3.1367396445106176e+17, 'train_loss': 16.005410401558763, 'epoch': 10.0})

In [18]:
test_output = trainer.predict(
    test_dataset=tokenized_test_dataset
)

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [19]:
for k, v in test_output.metrics.items():
    print(f"{k}: {v}")

test_loss: 0.537351667881012
test_runtime: 72.1593
test_samples_per_second: 92.67
test_steps_per_second: 11.585


In [20]:
raw_preds = test_output.predictions

In [21]:
predicted_answers = tokenizer.batch_decode(raw_preds, skip_special_tokens=True)

In [22]:
test_results_df = pd.DataFrame({
    "question": test_df["question"].values,
    "context": test_df["context"].values,
    "ground_truth": test_df["answer"].astype(str).values,
    "predicted": [p.strip() for p in predicted_answers],
})

In [23]:
test_results_df["is_exact_match"] = (
    test_results_df["predicted"].str.strip().str.lower()
    == test_results_df["ground_truth"].str.strip().str.lower()
)

In [24]:
test_results_df[test_results_df["is_exact_match"] == True]

,question,context,ground_truth,predicted,is_exact_match
0,What are the title(s) of episodes directed by ...,No. overall No. in season Title Directed by Wr...,buffy vs. dracula,buffy vs. dracula,True
1,who was the top driver at the 1967 british gra...,Pos No Driver Constructor Laps Time/Retired Gr...,no answer,no answer,True
2,"What is Years, when Postition is PG, and when ...",Player Nationality Jersey Number(s) Position Y...,1977-1984,1977-1984,True
4,which rider listed is the only one not to ride...,Year Rider Victories Bike Manufacturer's Champ...,no answer,no answer,True
6,how many graduate students does yale have?,Institution . Location Athletic nickname Under...,no answer,no answer,True
...,...,...,...,...,...
6677,how many parties had more than 50 seats in the...,Party First Duma Second Duma Third Duma Fourth...,no answer,no answer,True
6679,how many titles did melbourne house publish,Title Year + Publisher = Developer . Platforms...,no answer,no answer,True
6683,What's the total percent of the $40-50K ones t...,Data ³³ Headcount Percentage of total Less tha...,46.5%,46.5%,True
6685,what is the score when the surface is clay and...,Legend Grand Slam 0 WTA Championships 0 Tier I...,"7-6, 7-5","7-6, 7-5",True


In [25]:
test_results_df[test_results_df["is_exact_match"] == False]

,question,context,ground_truth,predicted,is_exact_match
3,How many attendance numbers had more than 15 g...,Season Record Pct. Games Attendance Average Se...,1,0,False
5,What is the latest year when Phil Anderson won?,Year Race name Rider status Winner Team/Countr...,1993,1991,False
7,"How many Played has a % Win of 50.00%, and Los...",Year Played Wins Losses No Result % Win T20 Cu...,0,no answer,False
10,What is the number for combaya municipality wh...,Language Sorata Municipality Guanay Municipali...,20.0,363,False
13,Which Away team played when they had a Crowd o...,Home team Home team score Away team Away team ...,richmond,collingwood,False
...,...,...,...,...,...
6678,Who is the GamePlan when Arabella Lazenby is t...,Actor in original production Actor required Ga...,lynette saxon,annette sefton-wilcox,False
6680,Mike Miller played for what school/club team?,Player No. Nationality Position Years in Orlan...,florida,florida cuttino mobley,False
6681,What is the total number of goals that have ga...,Club * AFL debuts Change club . Adelaide 6 2 B...,1,0,False
6682,What tournament had a Runner(s)-up of Tom Kite?,No. Year Tournament Opponent(s) Result 1 1987 ...,charles schwab cup championship,bank of america championship,False


In [26]:
test_results_df[test_results_df["is_exact_match"] == False]['context'][6678]

'Actor in original production Actor required GamePlan FlatSpin RolePlay Bill Champion Male, younger Troy Stephens Sam Berryman Justin Lazenby Saskia Butler Female, younger Sorrel Saxon Tracy Taylor Julie-Ann Jobson Alison Pargeter Female, younger Kelly Butcher Rosie Seymour Paige Petite Tim Faraday Male, older Dan Endicott Tommy Angel Micky Rale Robert Austin Male, older Leo Tyler Maurice Whickett Derek Jobson Beth Tuckley Female, older Grace Page Edna Stricken Dee Jobson Jacqueline King Female, older Lynette Saxon Annette Sefton- Wilcox Arabella Lazenby'

In [27]:
em_metric = evaluate.load("exact_match")

results = em_metric.compute(
    predictions=test_df["answer"].astype(str).values,
    references=[p.strip() for p in predicted_answers],
    ignore_case=True,
    ignore_punctuation=True,
)

In [28]:
print(f"EM: {results['exact_match'] * 100:.2f}%")

EM: 67.25%
